# Cashierless Vision — Detector Training on Kaggle GPU

Trains the YOLO/RT-DETR product/person detector for the cashierless-vision project
on a **free Kaggle GPU** (NVIDIA P100 16 GB or T4 x2), so you don't train on your
laptop. Produces `best.pt` + an ONNX export you can pull back into the repo and
feed to `make export` for TensorRT/Triton.

### Before you run
1. **Settings (right panel) → Accelerator → GPU** (P100 or T4 x2).
2. **Settings → Internet → On** (needed for pip + weight downloads; requires a
   phone-verified Kaggle account).
3. Run cells top to bottom.

GPU quota is ~30 hours/week with a 12-hour session cap — plenty for a detector.
With no dataset attached, the notebook runs a 3-epoch **smoke test** on `coco8`
just to prove the GPU pipeline works end to end.

## 1. Environment & GPU check

In [ ]:
# Update Ultralytics (Kaggle's preinstalled version is often old) + ONNX export deps.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "ultralytics", "onnx", "onnxslim", "onnxruntime"], check=False)

import torch, ultralytics
print("ultralytics:", ultralytics.__version__)
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| devices:", torch.cuda.device_count())
else:
    print("WARNING: no GPU — enable it in Settings -> Accelerator.")


In [ ]:
!nvidia-smi

## 2. Dataset

Three ways to feed data, in order of preference:

1. **Attach a Kaggle dataset** (Add Input, right panel) — a YOLO-format export with
   a `data.yaml`. Set `DATASET_YAML` to its path, e.g.
   `/kaggle/input/your-retail-set/data.yaml`.
2. **Roboflow / public retail-product set** — export in YOLOv8 format, upload as a
   Kaggle dataset, point `DATASET_YAML` at its `data.yaml`.
3. **Smoke test** — leave `DATASET_YAML = ""` to validate the pipeline on `coco8`.

Your `data.yaml` `names:` must match the store scheme (order matters):
`0: person, 1: product, 2: cart, 3: hand`.

In [ ]:
from pathlib import Path

# ============== CONFIG — edit these ==============
MODEL        = "yolov10m.pt"   # matches repo's detector_yolo ("yolo11m.pt" = newer alt)
DATASET_YAML = ""              # "/kaggle/input/<set>/data.yaml"  |  "" = smoke test
EPOCHS       = 100
IMGSZ        = 640
BATCH        = 16              # 16 is safe on 16GB at imgsz 640; lower if you hit OOM
CLASSES      = ["person", "product", "cart", "hand"]
PROJECT_DIR  = "/kaggle/working/runs"
# =================================================

# Smoke-test fallback so the GPU path is verifiable before real data exists.
if DATASET_YAML and Path(DATASET_YAML).exists():
    DATA, EPOCHS_EFF, SMOKE = DATASET_YAML, EPOCHS, False
else:
    print("No dataset -> SMOKE TEST on coco8 (auto-downloaded). NOT a real model.")
    DATA, EPOCHS_EFF, SMOKE = "coco8.yaml", min(EPOCHS, 3), True

print(f"dataset={DATA}  epochs={EPOCHS_EFF}  smoke={SMOKE}")


## 3. Train

In [ ]:
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else "cpu"
model = YOLO(MODEL)

results = model.train(
    data=DATA,
    epochs=EPOCHS_EFF,
    imgsz=IMGSZ,
    batch=BATCH,
    device=device,
    project=PROJECT_DIR,
    name="train",
    exist_ok=True,
    patience=20,        # early stop if val plateaus
    # AMP/FP16 is on by default — ideal for P100/T4.
)
SAVE_DIR = Path(model.trainer.save_dir)
print("Saved to:", SAVE_DIR)


## 4. Validate

In [ ]:
metrics = model.val()
print("mAP50    :", round(float(metrics.box.map50), 4))
print("mAP50-95 :", round(float(metrics.box.map), 4))


## 5. Export to ONNX

Export ONNX here. **Build the TensorRT `.plan` on your deployment GPU** (the Triton
box), not on Kaggle — TensorRT engines are hardware-specific and won't transfer.
Back in the repo, `make export` does ONNX→TensorRT→Triton on the target machine.

In [ ]:
onnx_path = model.export(format="onnx", imgsz=IMGSZ, half=True,
                         dynamic=True, simplify=True)
print("ONNX:", onnx_path)


## 6. (Optional) MLflow tracking
Mirrors the repo's MLflow setup. The sqlite DB lands in `/kaggle/working` so you can download and merge it locally. Safe to skip.

In [ ]:
try:
    import mlflow
    mlflow.set_tracking_uri("sqlite:////kaggle/working/mlflow.db")
    mlflow.set_experiment("store_product_detection")
    with mlflow.start_run():
        mlflow.log_params({"model": MODEL, "epochs": EPOCHS_EFF, "imgsz": IMGSZ,
                           "batch": BATCH, "dataset": DATA, "smoke_test": SMOKE})
        mlflow.log_metric("mAP50", float(metrics.box.map50))
        mlflow.log_metric("mAP50_95", float(metrics.box.map))
        best = SAVE_DIR / "weights" / "best.pt"
        if best.exists():
            mlflow.log_artifact(str(best), artifact_path="weights")
    print("Logged to /kaggle/working/mlflow.db")
except Exception as e:
    print("MLflow logging skipped:", e)


## 7. Package weights for download

In [ ]:
import shutil

out = Path("/kaggle/working/cashierless_artifacts")
out.mkdir(exist_ok=True)
for f in (SAVE_DIR / "weights").glob("*"):      # best.pt, last.pt, *.onnx
    shutil.copy(f, out / f.name)
shutil.make_archive("/kaggle/working/cashierless_artifacts", "zip", out)
print("Files:", [p.name for p in out.iterdir()])
print("Download /kaggle/working/cashierless_artifacts.zip from the Output tab ->")


## 8. Bring the model back into the repo

1. Download `cashierless_artifacts.zip` from the **Output** tab (right panel).
2. Put `best.pt` where the repo expects it, e.g. `runs/train/weights/best.pt`
   (or `models/production/best.pt` to make it the promotion-gate incumbent).
3. On your **deployment GPU**, build the TensorRT engine and stage it into Triton:
   ```bash
   make export        # ONNX -> TensorRT .plan -> deployment/triton_model_repository/detector_yolo/1/
   ```
4. Verify the class order in your training `data.yaml` matches
   `["person","product","cart","hand"]` so `triton_client.py` maps ids correctly.

That's the full handoff: train on Kaggle's GPU, serve the optimized engine from
your own Triton box.